In [1]:
import sys
sys.path.append('/home/ec2-user/SEAJ-TEAM-4-')
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np
from IPython.display import display


In [ ]:
from backend.data_processing.constants import DEFAULT_TICKERS
from backend.data_analysis.loader import MarketDataLoader
from backend.data_analysis.cleaner import MarketDataCleaner
from backend.data_analysis.features import MarketFeatureEngineer

loader = MarketDataLoader()
raw_df = loader.load_analysis_data(DEFAULT_TICKERS)

print("Raw Shape:", raw_df.shape)
clean_df = MarketDataCleaner.clean(raw_df)
df = MarketFeatureEngineer.add_all_features(clean_df)


2026-09-15 08:54:35,577 - INFO - Starting data load for 47 tickers
2026-09-15 08:54:35,578 - INFO - Downloading batch 1 with 47 tickers
2026-09-15 08:54:35,579 - INFO - Downloading historical data for 47 instruments from 2016-01-01 to None
2026-09-15 08:54:39,754 - INFO - Successfully downloaded data for batch of 47 tickers
2026-09-15 08:54:40,073 - INFO - Converted to long format: 126383 records, 7 columns
2026-09-15 08:54:40,076 - INFO - Successfully downloaded data for 126383 records across all batches


In [ ]:
from backend.data_analysis.analysis import(
    risk_return_summary,
    asset_class_summary,
    drawdown_summary
)

## Build asset class summary 

In [ ]:
risk_return = risk_return_summary(df)
drawdown_stats = drawdown_summary(df)
asset_summary = asset_class_summary(risk_return)

display(asset_summary)

## Average annual return by asset class

In [ ]:
return_table = asset_summary[[
    "asset_class",
    "avg_annual_return",
    "median_annual_return",
    "instrument_count",
    "return_label"
]].sort_values(by="avg_annual_return", ascending=False)
display(return_table)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(return_table["asset_class"], return_table["avg_annual_return"], color="steelblue")
ax.set_title("Average Annual Return by Asset Class")
ax.set_xlabel("Asset Class")
ax.set_ylabel("Average Annual Return")
plt.xticks(rotation=30)
fig.tight_layout()
plt.show()

## Average annual volatility by asset class

In [ ]:
volatility_table = asset_summary[[
    "asset_class",
    "avg_annual_volatility",
    "median_annual_volatility",
    "instrument_count",
    "risk_label"
]].sort_values(by="avg_annual_volatility", ascending=False)
display(volatility_table)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(volatility_table["asset_class"], volatility_table["avg_annual_volatility"], color="indianred")
ax.set_title("Average Annual Volatility by Asset Class")
ax.set_xlabel("Asset Class")
ax.set_ylabel("Average Annual Volatility")
plt.xticks(rotation=30)
fig.tight_layout()
plt.show()

## Mean vs Median return 

In [ ]:
mean_median_table = asset_summary[[
    "asset_class",
    "avg_annual_return",
    "median_annual_return",
    "avg_annual_volatility",
    "median_annual_volatility"
]].copy()
mean_median_table["return_gap"] = mean_median_table["avg_annual_return"] - mean_median_table["median_annual_return"]
mean_median_table["volatility_gap"] = mean_median_table["avg_annual_volatility"] - mean_median_table["median_annual_volatility"]
display(mean_median_table.sort_values(by="return_gap", ascending=False))

positions = np.arange(len(mean_median_table))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(positions - width / 2, mean_median_table["avg_annual_return"], width=width, label="Mean Annual Return")
ax.bar(positions + width / 2, mean_median_table["median_annual_return"], width=width, label="Median Annual Return")
ax.set_xticks(positions)
ax.set_xticklabels(mean_median_table["asset_class"], rotation=30)
ax.set_title("Mean vs Median Annual Return by Asset Class")
ax.set_xlabel("Asset Class")
ax.set_ylabel("Annual Return")
ax.legend()
fig.tight_layout()
plt.show()

## Asset Class drawdown

In [ ]:
asset_drawdown = (
    drawdown_stats.groupby("asset_class")
    .agg(
        avg_max_drawdown_pct=("max_drawdown_pct", "mean"),
        median_max_drawdown_pct=("max_drawdown_pct", "median"),
        avg_drawdown_pct=("avg_drawdown_pct", "mean"),
        instrument_count=("symbol", "count")
    )
    .reset_index()
    .sort_values(by="avg_max_drawdown_pct")
)
display(asset_drawdown)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(asset_drawdown["asset_class"], asset_drawdown["avg_max_drawdown_pct"], color="darkorange")
ax.set_title("Average Maximum Drawdown by Asset Class")
ax.set_xlabel("Average Maximum Drawdown (%)")
ax.set_ylabel("Asset Class")
fig.tight_layout()
plt.show()

## Risk/return position of each asset class

In [ ]:
risk_return_position = asset_summary[[
    "asset_class",
    "avg_annual_return",
    "avg_annual_volatility",
    "instrument_count",
    "return_label",
    "risk_label"
]].sort_values(by="avg_annual_return", ascending=False)
display(risk_return_position)

fig, ax = plt.subplots(figsize=(10, 6))
sizes = risk_return_position["instrument_count"] * 80
ax.scatter(
    risk_return_position["avg_annual_volatility"],
    risk_return_position["avg_annual_return"],
    s=sizes,
    alpha=0.7,
    color="mediumpurple"
)
for _, row in risk_return_position.iterrows():
    ax.annotate(
        row["asset_class"],
        (row["avg_annual_volatility"], row["avg_annual_return"]),
        xytext=(5, 5),
        textcoords="offset points"
    )
ax.set_title("Risk and Return Position by Asset Class")
ax.set_xlabel("Average Annual Volatility")
ax.set_ylabel("Average Annual Return")
ax.axhline(0, color="grey", linewidth=1)
fig.tight_layout()
plt.show()